In [ ]:
import cirq
import sympy
import numpy as np
from cirq.contrib.svg import SVGCircuit

def lih_fig13_circuit(
    theta1: sympy.Symbol, theta2: sympy.Symbol, theta3: sympy.Symbol
) -> tuple[cirq.Circuit, list[cirq.LineQubit]]:
    """Compiled LiH ansatz (FIG. 13): three independent ``RX`` angles on ``q[1]``.

    """
    q = cirq.LineQubit.range(6)
    q0, q1, q2, q3, q4, q5 = q
    c = cirq.Circuit()
    ## this is for OG HF state
    # c.append(cirq.X(q0))
    # c.append(cirq.X(q3))


    c.append(cirq.rx(np.pi / 2).on(q0))
    c.append(cirq.H(q2))
    c.append(cirq.H(q3))
    c.append(cirq.H(q4))
    c.append(cirq.H(q5))

    c.append([cirq.CZ(q0, q1), cirq.CZ(q3, q4)])
    c.append([cirq.CZ(q4, q5), cirq.H(q4)])

    c.append(cirq.CZ(q1, q4))
    c.append(cirq.rx(theta1).on(q1))
    c.append(cirq.CZ(q1, q4))

    c.append(cirq.CZ(q0, q1))
    c.append(cirq.H(q1))
    c.append(cirq.CZ(q0, q1))
    c.append(cirq.CZ(q1, q2))

    c.append(cirq.CZ(q1, q4))
    c.append(cirq.rx(theta2).on(q1))
    c.append(cirq.CZ(q1, q4))

    c.append(cirq.H(q4))
    c.append(cirq.CZ(q4, q5))
    c.append(cirq.H(q5))
    c.append(cirq.CZ(q3, q4))
    c.append(cirq.H(q4))
    c.append(cirq.CZ(q3, q4))
    c.append(cirq.H(q4))

    c.append(cirq.CZ(q1, q4))
    c.append(cirq.rx(theta3).on(q1))
    c.append(cirq.CZ(q1, q4))

    c.append(cirq.H(q4))
    c.append(cirq.CZ(q1, q2))
    c.append(cirq.H(q2))
    c.append(cirq.CZ(q0, q1))
    c.append(cirq.CZ(q3, q4))
    c.append(cirq.rx(-np.pi / 2).on(q0))
    c.append([cirq.H(q1), cirq.H(q3)])

    return c, q


theta1 = sympy.Symbol("theta1")
theta2 = sympy.Symbol("theta2")
theta3 = sympy.Symbol("theta3")
circuit, qubits = lih_fig13_circuit(theta1, theta2, theta3)

# Diagram with θ₁ = θ₂ = θ₃ = π/5 (paper-style visualization). HF check uses all zeros in the next cell.
_theta_viz = np.pi / 5
resolver_viz = cirq.ParamResolver({theta1: _theta_viz, theta2: _theta_viz, theta3: _theta_viz})
circuit_draw = cirq.resolve_parameters(circuit, resolver_viz)

print(circuit_draw)
display(SVGCircuit(circuit_draw))

In [ ]:
import cirq
import sympy
import numpy as np
from cirq.contrib.svg import SVGCircuit

class UCCSDCompiler:
    def __init__(self, qubits):
        self.qubits = qubits
        self.circuit = cirq.Circuit()
        self.current_basis = {i: 'Z' for i in range(len(qubits))}

    def apply_basis_change(self, target_pauli_dict):
        for idx, target_pauli in target_pauli_dict.items():
            current_pauli = self.current_basis[idx]
            if current_pauli == target_pauli:
                continue

            q = self.qubits[idx]
            if current_pauli == 'X': self.circuit.append(cirq.H(q))
            elif current_pauli == 'Y': self.circuit.append(cirq.rx(-np.pi / 2).on(q))

            if target_pauli == 'X': self.circuit.append(cirq.H(q))
            elif target_pauli == 'Y': self.circuit.append(cirq.rx(np.pi / 2).on(q))

            self.current_basis[idx] = target_pauli

    def append_term(self, pauli_string, theta):
        self.apply_basis_change(pauli_string)
        active_indices = sorted(list(pauli_string.keys()))

        # 1. CNOT Ladder down
        for i in range(len(active_indices) - 1):
            c, t = self.qubits[active_indices[i]], self.qubits[active_indices[i+1]]
            self.circuit.append([cirq.H(t), cirq.CZ(c, t), cirq.H(t)])

        # 2. Phase Rotation
        target = self.qubits[active_indices[-1]]
        self.circuit.append(cirq.rz(2 * theta).on(target))

        # 3. CNOT Ladder up
        for i in reversed(range(len(active_indices) - 1)):
            c, t = self.qubits[active_indices[i]], self.qubits[active_indices[i+1]]
            self.circuit.append([cirq.H(t), cirq.CZ(c, t), cirq.H(t)])

    def _clean_hh_gates(self):
        """Custom Peephole Optimizer: Safely cancels adjacent H gates on the same wire."""
        clean_circuit = cirq.Circuit()
        # Keep track of whether an H gate is "waiting" to be placed on a wire
        pending_h = {q: False for q in self.qubits}

        for moment in self.circuit:
            ops_to_add = []
            for op in moment:
                if op.gate == cirq.H:
                    q = op.qubits[0]
                    # If an H is already pending, they cancel out! Toggle to False.
                    pending_h[q] = not pending_h[q]
                else:
                    # For any other gate (like CZ or Rz), flush pending H gates first
                    for q in op.qubits:
                        if pending_h.get(q, False):
                            ops_to_add.append(cirq.H(q))
                            pending_h[q] = False
                    ops_to_add.append(op)

            if ops_to_add:
                # Add the moment, ignoring empty gaps
                clean_circuit.append(ops_to_add, strategy=cirq.InsertStrategy.NEW)

        # Flush any remaining H gates at the very end of the circuit
        flush_ops = [cirq.H(q) for q, has_h in pending_h.items() if has_h]
        if flush_ops:
            clean_circuit.append(flush_ops)

        self.circuit = clean_circuit

    def finalize(self):
        # 1. Return everything to Z basis
        self.apply_basis_change({i: 'Z' for i in range(len(self.qubits))})

        # 2. Run our custom, symbol-safe Peephole Optimizer
        self._clean_hh_gates()

        # 3. Compress visual gaps
        cirq.drop_empty_moments(self.circuit)

        return self.circuit


# --- Build & Visualize ---
q = cirq.LineQubit.range(6)
compiler = UCCSDCompiler(q)

theta1, theta2, theta3 = sympy.symbols("theta1 theta2 theta3")

compiler.append_term({0: 'Y', 1: 'X', 3: 'X', 4: 'Z', 5: 'X'}, theta1)
compiler.append_term({0: 'Y', 1: 'Z', 2: 'X', 3: 'X', 4: 'X'}, theta2)
compiler.append_term({0: 'Y', 1: 'Z', 2: 'X', 3: 'X', 4: 'Z', 5: 'X'}, theta3)

optimized_circuit = compiler.finalize()

# Resolve and Draw
_theta_viz = np.pi / 3
resolver_viz = cirq.ParamResolver({theta1: _theta_viz, theta2: _theta_viz, theta3: _theta_viz})
circuit_draw = cirq.resolve_parameters(optimized_circuit, resolver_viz)

display(SVGCircuit(circuit_draw))

In [9]:
# pip install openfermion
from openfermion import QubitOperator

# =====================================================================
# 1. Define the fundamental Jordan-Wigner mappings from scratch
# =====================================================================
def adag(j):
    """Creation operator: a^\dagger_j = 1/2(X_j - iY_j) Z_{j-1}...Z_0"""
    op = QubitOperator(f'X{j}', 0.5) + QubitOperator(f'Y{j}', -0.5j)
    # Apply the Z string to all lower-indexed qubits
    for i in range(j):
        op = QubitOperator(f'Z{i}') * op 
    return op

def a(j):
    """Annihilation operator: a_j = 1/2(X_j + iY_j) Z_{j-1}...Z_0"""
    op = QubitOperator(f'X{j}', 0.5) + QubitOperator(f'Y{j}', 0.5j)
    # Apply the Z string to all lower-indexed qubits
    for i in range(j):
        op = QubitOperator(f'Z{i}') * op 
    return op

# =====================================================================
# 2. Construct the UCCSD Generator (T - T^\dagger)
# =====================================================================
# Define the specific double excitation T = a^\dagger_3 a^\dagger_1 a_2 a_0
T = adag(3) * adag(1) * a(2) * a(0)

# Define the Hermitian conjugate: (ABCD)^\dagger = D^\dagger C^\dagger B^\dagger A^\dagger
T_dag = adag(0) * adag(2) * a(1) * a(3)

# Calculate the anti-Hermitian generator
# This symbolically expands the (X - iY) multiplication and cancels the real parts
generator = T - T_dag

print("--- 1. Full Jordan-Wigner Expansion ---")
print("The 8 purely imaginary Pauli strings generated by T - T^\dagger:\n")
print(generator)


# =====================================================================
# 3. Analyze the Redundancy on the Hartree-Fock State |1100>
# =====================================================================
print("\n--- 2. Redundancy Analysis on |1100> ---")

# Define our starting state: Qubits 0 and 1 are occupied (1), 2 and 3 are empty (0).
hf_state = {0: 1, 1: 0, 2: 1, 3: 0}
print(f"Initial Hartree-Fock state: {hf_state}\n")

def apply_pauli_string(pauli_dict, state_dict):
    """Simulates the action of a single Pauli string on a basis state."""
    new_state = dict(state_dict)
    phase = 1.0 + 0.0j

    for qubit, pauli in pauli_dict.items():
        bit = new_state[qubit]
        if pauli == 'X':
            new_state[qubit] = 1 - bit           # X flips the bit
        elif pauli == 'Y':
            new_state[qubit] = 1 - bit           # Y flips the bit...
            phase *= 1j if bit == 0 else -1j     # ...and adds a phase
        elif pauli == 'Z':
            phase *= 1 if bit == 0 else -1       # Z adds phase if bit is 1

    return new_state, phase

# Iterate through the 8 terms to see what they actually do to the state
for term, coefficient in generator.terms.items():
    # Convert OpenFermion tuple formatting e.g., ((0, 'X'), (1, 'Y')) to a dictionary
    pauli_dict = {q: p for q, p in term}
    
    # Apply this specific Pauli string to |1100>
    resulting_state, phase = apply_pauli_string(pauli_dict, hf_state)
    
    # Calculate the total physical contribution (coefficient * Pauli phase)
    total_amplitude = coefficient * phase
    
    print(f"Applying Operator: {pauli_dict}")
    print(f" -> Result: {resulting_state} | Amplitude Contribution: {total_amplitude}")

--- 1. Full Jordan-Wigner Expansion ---
The 8 purely imaginary Pauli strings generated by T - T^\dagger:

0.125j [X0 X1 X2 Y3] +
(-0-0.125j) [X0 X1 Y2 X3] +
0.125j [X0 Y1 X2 X3] +
0.125j [X0 Y1 Y2 Y3] +
(-0-0.125j) [Y0 X1 X2 X3] +
(-0-0.125j) [Y0 X1 Y2 Y3] +
0.125j [Y0 Y1 X2 Y3] +
(-0-0.125j) [Y0 Y1 Y2 X3]

--- 2. Redundancy Analysis on |1100> ---
Initial Hartree-Fock state: {0: 1, 1: 0, 2: 1, 3: 0}

Applying Operator: {0: 'X', 1: 'X', 2: 'Y', 3: 'X'}
 -> Result: {0: 0, 1: 1, 2: 0, 3: 1} | Amplitude Contribution: (-0.125+0j)
Applying Operator: {0: 'Y', 1: 'X', 2: 'X', 3: 'X'}
 -> Result: {0: 0, 1: 1, 2: 0, 3: 1} | Amplitude Contribution: (-0.125+0j)
Applying Operator: {0: 'Y', 1: 'Y', 2: 'Y', 3: 'X'}
 -> Result: {0: 0, 1: 1, 2: 0, 3: 1} | Amplitude Contribution: (-0.125+0j)
Applying Operator: {0: 'X', 1: 'Y', 2: 'X', 3: 'X'}
 -> Result: {0: 0, 1: 1, 2: 0, 3: 1} | Amplitude Contribution: (-0.125+0j)
Applying Operator: {0: 'Y', 1: 'X', 2: 'Y', 3: 'Y'}
 -> Result: {0: 0, 1: 1, 2: 0, 3: 1}